In [1]:
!pip install torch torchvision torchaudio --quiet

In [2]:
!pip install numpy pandas scikit-learn matplotlib --quiet

In [3]:
!pip install PyEMD --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.6 MB/s eta 0:00:00


In [4]:
import os
import csv
import math
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler   # ← AMP
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
DATA_PATH   = "/content/drive/MyDrive/ceemdan_components.csv"   # adjust path if needed
OUTPUT_CSV  = "/content/drive/MyDrive/transformer_experiment_results.csv"
OUTPUT_TXT  = "/content/drive/MyDrive/transformer_experiment_summary.txt"

# Component lists
IMF_COLS      = ["IMF1", "IMF2", "IMF3", "IMF4", "IMF5", "IMF6", "IMF7", "IMF8"]
RESIDUAL_COL  = "Residual"
TARGET_COL    = "Close"
# IMF8 uses first-diff + StandardScaler; all others use MinMaxScaler
MINMAX_COMPS  = ["IMF1", "IMF2", "IMF3", "IMF4", "IMF5", "IMF6", "IMF7", "Residual"]
ALL_COMPONENTS = IMF_COLS + [RESIDUAL_COL]   # 9 total

# ── Transformer Hyperparameters ────────────────────────────────────────────────
T_SEQ_LEN    = 96
T_LABEL_LEN  = 48
T_PRED_LEN   = 1

T_D_MODEL    = 256
T_N_HEADS    = 8
T_E_LAYERS   = 2      # encoder layers
T_D_LAYERS   = 1      # decoder layers

T_D_FF       = 1024
T_DROPOUT    = 0.1
T_ACTIVATION = "relu"  # used in feed-forward sub-layers

T_BATCH      = 32
T_EPOCHS     = 100
T_LR         = 0.0001
T_PATIENCE   = 10      # early stopping patience (adaptive; not in param dict)

# ── Experiment settings ────────────────────────────────────────────────────────
N_EXPERIMENTS      = 50          # seeds 1 → 50
ROLLING_VOL_WINDOW = 30

# Checkpoint — reuse output path so a resume picks up where it left off
CHECKPOINT_CSV = OUTPUT_CSV

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [7]:
def set_seed(seed: int):
    """Fully deterministic seeding for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


In [8]:
def create_transformer_windows(series: np.ndarray,
                                seq_len: int, label_len: int, pred_len: int):
    """
    Sliding-window generator for an encoder-decoder Transformer (pred_len=1).

    Encoder input  : series[i : i+seq_len]            shape → (seq_len, 1)
    Decoder input  : series[i+seq_len-label_len : i+seq_len]
                     concatenated with zeros(pred_len) shape → (label_len+pred_len, 1)
    Target         : series[i+seq_len]                 scalar

    Returns
    -------
    enc_x   : (N, seq_len,              1)  float32
    dec_x   : (N, label_len + pred_len, 1)  float32
    targets : (N,)                           float32
    """
    enc_x, dec_x, targets = [], [], []
    for i in range(len(series) - seq_len - pred_len + 1):
        enc_seq    = series[i : i + seq_len]
        label_part = series[i + seq_len - label_len : i + seq_len]
        pred_part  = np.zeros(pred_len, dtype=np.float32)
        dec_seq    = np.concatenate([label_part, pred_part])
        target_val = series[i + seq_len]
        enc_x.append(enc_seq)
        dec_x.append(dec_seq)
        targets.append(target_val)
    enc_x   = np.array(enc_x,   dtype=np.float32)[..., np.newaxis]
    dec_x   = np.array(dec_x,   dtype=np.float32)[..., np.newaxis]
    targets = np.array(targets, dtype=np.float32)
    return enc_x, dec_x, targets



In [9]:
class TransformerDataset(Dataset):
    def __init__(self, enc_x, dec_x, targets):
        self.enc_x   = torch.from_numpy(enc_x)
        self.dec_x   = torch.from_numpy(dec_x)
        self.targets = torch.from_numpy(targets)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.enc_x[idx], self.dec_x[idx], self.targets[idx]

In [10]:
class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding (Vaswani et al., 2017)."""
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)                     # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ── Feed-Forward Sub-layer ───────────────────────────────────────────────────
def _make_ff(d_model: int, d_ff: int, dropout: float, activation: str):
    """Create a two-layer FFN with the specified activation (relu / gelu)."""
    act = nn.ReLU() if activation.lower() == "relu" else nn.GELU()
    return nn.Sequential(
        nn.Linear(d_model, d_ff),
        act,
        nn.Dropout(dropout),
        nn.Linear(d_ff, d_model),
    )


# ── Encoder Layer ─────────────────────────────────────────────────────────────
class TransformerEncoderLayer(nn.Module):
    """
    Standard Transformer encoder layer.
    Self-attention  → Add & Norm → FFN → Add & Norm
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 dropout: float, activation: str):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True)
        self.ff    = _make_ff(d_model, d_ff, dropout, activation)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


# ── Decoder Layer ─────────────────────────────────────────────────────────────
class TransformerDecoderLayer(nn.Module):
    """
    Standard Transformer decoder layer.
    Masked self-attention (causal)  → Add & Norm
    Cross-attention with encoder    → Add & Norm
    FFN                             → Add & Norm
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 dropout: float, activation: str):
        super().__init__()
        self.self_attn  = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True)
        self.ff    = _make_ff(d_model, d_ff, dropout, activation)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor,
                tgt_mask: torch.Tensor = None) -> torch.Tensor:
        # Masked self-attention (causal mask prevents looking ahead)
        sa_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = self.norm1(x + self.drop(sa_out))
        # Cross-attention: queries from decoder, keys/values from encoder
        ca_out, _ = self.cross_attn(x, enc_out, enc_out)
        x = self.norm2(x + self.drop(ca_out))
        x = self.norm3(x + self.drop(self.ff(x)))
        return x


# ── Full Vanilla Transformer ───────────────────────────────────────────────────
class VanillaTransformer(nn.Module):
    """
    Encoder-Decoder Transformer for next-step forecasting.

    Input shapes:
        enc_x : (B, seq_len,              1)
        dec_x : (B, label_len + pred_len, 1)
    Output:
        (B,)  — single scalar prediction per sample
    """
    def __init__(self,
                 d_model    : int   = T_D_MODEL,
                 n_heads    : int   = T_N_HEADS,
                 e_layers   : int   = T_E_LAYERS,
                 d_layers   : int   = T_D_LAYERS,
                 d_ff       : int   = T_D_FF,
                 dropout    : float = T_DROPOUT,
                 activation : str   = T_ACTIVATION,
                 seq_len    : int   = T_SEQ_LEN,
                 label_len  : int   = T_LABEL_LEN,
                 pred_len   : int   = T_PRED_LEN):
        super().__init__()
        self.pred_len  = pred_len
        self.label_len = label_len

        # Input projections: univariate (1-D) → d_model
        self.enc_embed = nn.Linear(1, d_model)
        self.dec_embed = nn.Linear(1, d_model)

        # Positional encodings
        self.enc_pos   = PositionalEncoding(
            d_model, max_len=seq_len + 10, dropout=dropout)
        self.dec_pos   = PositionalEncoding(
            d_model, max_len=label_len + pred_len + 10, dropout=dropout)

        # Encoder stack
        self.encoder   = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, activation)
            for _ in range(e_layers)])

        # Decoder stack
        self.decoder   = nn.ModuleList([
            TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, activation)
            for _ in range(d_layers)])

        self.enc_norm  = nn.LayerNorm(d_model)
        self.dec_norm  = nn.LayerNorm(d_model)

        # Output projection: d_model → 1 scalar
        self.proj      = nn.Linear(d_model, 1)

    @staticmethod
    def _causal_mask(sz: int, device: torch.device) -> torch.Tensor:
        """
        Upper-triangular causal mask for decoder self-attention.
        True  → masked (ignored); False → attended.
        Shape: (sz, sz)
        """
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return mask

    def forward(self, enc_x: torch.Tensor,
                dec_x: torch.Tensor) -> torch.Tensor:
        # ── Encoder ──────────────────────────────────────────────────────────
        enc_out = self.enc_pos(self.enc_embed(enc_x))   # (B, seq_len, d_model)
        for layer in self.encoder:
            enc_out = layer(enc_out)
        enc_out = self.enc_norm(enc_out)

        # ── Decoder ──────────────────────────────────────────────────────────
        dec_out = self.dec_pos(self.dec_embed(dec_x))   # (B, dec_len, d_model)
        dec_len = dec_out.size(1)
        tgt_mask = self._causal_mask(dec_len, dec_out.device)
        for layer in self.decoder:
            dec_out = layer(dec_out, enc_out, tgt_mask)
        dec_out = self.dec_norm(dec_out)

        # Take the last pred_len positions and project to scalar
        out = self.proj(dec_out[:, -self.pred_len:, :])   # (B, pred_len, 1)
        return out.squeeze(-1).squeeze(-1)                  # (B,)

In [11]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Return MAE, RMSE, MAPE, R² computed on original-scale arrays."""
    y_true = np.array(y_true, dtype=np.float64)
    y_pred = np.array(y_pred, dtype=np.float64)
    mae  = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mask = np.abs(y_true) > 1e-8
    mape = float(np.mean(np.abs(
        (y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
    r2   = float(r2_score(y_true, y_pred))
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}

In [12]:
def _precompute_component_windows(train_sc, val_sc, test_sc):
    """
    Build and return GPU-resident tensors for one component.
    Called once per component (not once per seed).

    Returns
    -------
    tr_gpu  : (enc_tr, dec_tr, y_tr) tensors on DEVICE
    vl_gpu  : (enc_vl, dec_vl, y_vl) tensors on DEVICE
    te_enc  : enc_te tensor on DEVICE
    te_dec  : dec_te tensor on DEVICE
    """
    val_context  = np.concatenate([train_sc[-T_SEQ_LEN:], val_sc])
    test_context = np.concatenate([val_sc[-T_SEQ_LEN:],   test_sc])

    enc_tr, dec_tr, y_tr = create_transformer_windows(
        train_sc,     T_SEQ_LEN, T_LABEL_LEN, T_PRED_LEN)
    enc_vl, dec_vl, y_vl = create_transformer_windows(
        val_context,  T_SEQ_LEN, T_LABEL_LEN, T_PRED_LEN)
    enc_te, dec_te, _    = create_transformer_windows(
        test_context, T_SEQ_LEN, T_LABEL_LEN, T_PRED_LEN)

    if len(enc_tr) == 0:
        raise ValueError("Training set too short for Transformer windows.")

    # Move to GPU once — reused for all 50 seeds
    to = dict(device=DEVICE, non_blocking=True)
    tr_gpu = (torch.from_numpy(enc_tr).to(**to),
              torch.from_numpy(dec_tr).to(**to),
              torch.from_numpy(y_tr).to(**to))
    vl_gpu = (torch.from_numpy(enc_vl).to(**to),
              torch.from_numpy(dec_vl).to(**to),
              torch.from_numpy(y_vl).to(**to))
    te_enc = torch.from_numpy(enc_te).to(**to)
    te_dec = torch.from_numpy(dec_te).to(**to)
    return tr_gpu, vl_gpu, te_enc, te_dec


def _train_transformer_fast(tr_gpu, vl_gpu, te_enc, te_dec,
                             patience: int = T_PATIENCE) -> np.ndarray:
    """
    Train a fresh VanillaTransformer using pre-built GPU tensors.
    Uses AMP (FP16) for ~2× GPU throughput on T4.

    Parameters
    ----------
    tr_gpu / vl_gpu : (enc, dec, y) tuples already on DEVICE
    te_enc / te_dec : test encoder/decoder inputs already on DEVICE
    patience        : early-stopping patience

    Returns
    -------
    test_predictions_scaled : 1-D float32 numpy array
    """
    enc_tr, dec_tr, y_tr = tr_gpu
    enc_vl, dec_vl, y_vl = vl_gpu
    n_tr = enc_tr.shape[0]
    n_vl = enc_vl.shape[0]
    n_te = te_enc.shape[0]

    use_amp = (DEVICE.type == "cuda")
    amp_scaler = GradScaler(enabled=use_amp)

    model  = VanillaTransformer().to(DEVICE)
    opt    = Adam(model.parameters(), lr=T_LR)
    crit   = nn.MSELoss()
    best_v = float("inf")
    best_w = None
    no_imp = 0

    # ── Training ──────────────────────────────────────────────────────────────
    for epoch in range(1, T_EPOCHS + 1):
        model.train()
        for start in range(0, n_tr, T_BATCH):
            eb = enc_tr[start : start + T_BATCH]
            db = dec_tr[start : start + T_BATCH]
            yb = y_tr  [start : start + T_BATCH]
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=use_amp):
                loss = crit(model(eb, db), yb)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(opt)
            amp_scaler.update()

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        vl_losses = []
        with torch.no_grad(), autocast(enabled=use_amp):
            for start in range(0, n_vl, T_BATCH):
                eb = enc_vl[start : start + T_BATCH]
                db = dec_vl[start : start + T_BATCH]
                yb = y_vl  [start : start + T_BATCH]
                vl_losses.append(crit(model(eb, db), yb).item())
        vl_loss = float(np.mean(vl_losses))

        if vl_loss < best_v:
            best_v = vl_loss
            best_w = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    # ── Test inference ────────────────────────────────────────────────────────
    model.load_state_dict(best_w)
    model.eval()
    preds = []
    with torch.no_grad(), autocast(enabled=use_amp):
        for start in range(0, n_te, T_BATCH):
            eb = te_enc[start : start + T_BATCH]
            db = te_dec[start : start + T_BATCH]
            preds.append(model(eb, db).float().cpu().numpy())
    return np.concatenate(preds)   # 1-D float32

In [13]:
def precompute_all_windows(df_train, df_val, df_test):
    """
    Fit scalers and build GPU-resident tensors for every component.
    Called ONCE before the 50-seed loop — not 50 times!

    Returns
    -------
    scalers      : dict  comp → fitted scaler (for inverse-transform)
    imf8_meta    : dict  {ss, anchor, tr_gpu, vl_gpu, te_enc, te_dec}
    comp_windows : dict  comp → (tr_gpu, vl_gpu, te_enc, te_dec)
    """
    print("  [Pre-compute] Fitting scalers & building GPU tensors …", flush=True)
    t0 = time.time()

    # ── MinMax components (IMF1–7 + Residual) ────────────────────────────────
    scalers      = {}
    comp_windows = {}
    for comp in MINMAX_COMPS:
        sc = MinMaxScaler(feature_range=(0, 1))
        sc.fit(df_train[[comp]].values)
        scalers[comp] = sc
        tr_sc = sc.transform(df_train[[comp]].values).flatten().astype(np.float32)
        vl_sc = sc.transform(df_val[[comp]].values).flatten().astype(np.float32)
        te_sc = sc.transform(df_test[[comp]].values).flatten().astype(np.float32)
        comp_windows[comp] = _precompute_component_windows(tr_sc, vl_sc, te_sc)
        print(f"    [{comp}] windowed", flush=True)

    # ── IMF8 first-diff pipeline ─────────────────────────────────────────────
    tr_raw = df_train["IMF8"].values.astype(np.float64)
    vl_raw = df_val["IMF8"].values.astype(np.float64)
    te_raw = df_test["IMF8"].values.astype(np.float64)
    d_tr   = np.diff(tr_raw)
    d_vl   = np.diff(np.concatenate([[tr_raw[-1]], vl_raw]))
    d_te   = np.diff(np.concatenate([[vl_raw[-1]], te_raw]))
    ss = StandardScaler()
    ss.fit(d_tr.reshape(-1, 1))
    d_tr_sc = ss.transform(d_tr.reshape(-1, 1)).flatten().astype(np.float32)
    d_vl_sc = ss.transform(d_vl.reshape(-1, 1)).flatten().astype(np.float32)
    d_te_sc = ss.transform(d_te.reshape(-1, 1)).flatten().astype(np.float32)
    imf8_meta = {
        "ss"     : ss,
        "anchor" : float(vl_raw[-1]),
        "windows": _precompute_component_windows(d_tr_sc, d_vl_sc, d_te_sc),
    }
    print(f"    [IMF8] windowed (first-diff)", flush=True)

    print(f"  [Pre-compute] Done in {time.time()-t0:.1f}s — "
          f"tensors live on {DEVICE}", flush=True)
    return scalers, imf8_meta, comp_windows


In [14]:
def run_experiment(scalers, imf8_meta, comp_windows,
                   df_test, seed: int) -> tuple:
    """
    Full CEEMDAN-Transformer pipeline for one seed.
    Receives pre-computed GPU tensors — only trains fresh models.

    Returns
    -------
    metrics  : dict  {MAE, RMSE, MAPE, R2, training_time_sec}
    comp_preds : dict  component → np.ndarray (original scale)
    y_true_a / y_hat_a : aligned ground-truth and predictions
    """
    set_seed(seed)
    t0 = time.time()
    comp_preds = {}

    # ── MinMax components ─────────────────────────────────────────────────────
    for comp in MINMAX_COMPS:
        tr_gpu, vl_gpu, te_enc, te_dec = comp_windows[comp]
        preds_sc = _train_transformer_fast(tr_gpu, vl_gpu, te_enc, te_dec)
        comp_preds[comp] = scalers[comp].inverse_transform(
            preds_sc.reshape(-1, 1)).flatten().astype(np.float64)
        print(f"    [{comp}] done", flush=True)

    # ── IMF8 first-diff ───────────────────────────────────────────────────────
    tr_gpu, vl_gpu, te_enc, te_dec = imf8_meta["windows"]
    delta_sc = _train_transformer_fast(tr_gpu, vl_gpu, te_enc, te_dec)
    delta    = imf8_meta["ss"].inverse_transform(
        delta_sc.reshape(-1, 1)).flatten().astype(np.float64)
    comp_preds["IMF8"] = imf8_meta["anchor"] + np.cumsum(delta)
    print(f"    [IMF8] done (first-diff)", flush=True)

    t_elapsed = time.time() - t0

    # ── Reconstruct: ŷ = Σ all 9 components ──────────────────────────────────
    n_test = len(df_test)
    y_hat  = np.zeros(n_test, dtype=np.float64)
    for comp in ALL_COMPONENTS:
        y_hat += comp_preds[comp][:n_test]

    y_true  = df_test[TARGET_COL].values.astype(np.float64)
    min_len = min(len(y_hat), len(y_true))
    y_hat_a  = y_hat[:min_len]
    y_true_a = y_true[:min_len]

    metrics = compute_metrics(y_true_a, y_hat_a)
    metrics["training_time_sec"] = t_elapsed
    return metrics, comp_preds, y_true_a, y_hat_a

In [15]:
_CKPT_FIELDS = ["experiment", "seed", "MAE", "RMSE", "MAPE", "R2",
                 "training_time_sec"]


def _load_checkpoint() -> tuple:
    """
    Read CHECKPOINT_CSV (if it exists) and return:
        completed_seeds : set of int  — seeds already finished
        all_results     : list of dict — one dict per completed seed
    """
    if not os.path.exists(CHECKPOINT_CSV):
        return set(), []
    try:
        ck = pd.read_csv(CHECKPOINT_CSV)
        if ck.empty:
            return set(), []
        records = ck.to_dict(orient="records")
        seeds   = {int(r["seed"]) for r in records}
        print(f"  ↻  Checkpoint found: {len(seeds)} seeds already done "
              f"({sorted(seeds)[0]}–{sorted(seeds)[-1]}).  Resuming…")
        return seeds, records
    except Exception as e:
        print(f"  ⚠  Could not read checkpoint ({e}). Starting fresh.")
        return set(), []


def _append_checkpoint(row: dict):
    """
    Append a single result row immediately after each seed completes.
    Creates the file with headers on the first write.
    """
    write_header = not os.path.exists(CHECKPOINT_CSV)
    with open(CHECKPOINT_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=_CKPT_FIELDS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        w.writerow(row)


def cell13_load_and_run():
    """Load dataset, split once, pre-compute windows once, run 50 experiments."""
    print(f"Device  : {DEVICE}")
    print(f"PyTorch : {torch.__version__}")
    print(f"AMP     : {'enabled (FP16)' if DEVICE.type == 'cuda' else 'disabled (CPU)'}")
    print(f"Running : {N_EXPERIMENTS} experiments  (seeds 1 → {N_EXPERIMENTS})")
    print("=" * 65)

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    train_mask    = df["Date"].dt.year <= 2022
    test_mask     = df["Date"].dt.year >= 2023
    df_train_full = df[train_mask].reset_index(drop=True)
    df_test       = df[test_mask].reset_index(drop=True)
    val_size      = int(len(df_train_full) * 0.10)
    df_train      = df_train_full.iloc[:-val_size].reset_index(drop=True)
    df_val        = df_train_full.iloc[-val_size:].reset_index(drop=True)

    print(f"Train : {len(df_train):>5}  "
          f"({df_train.Date.min().date()} → {df_train.Date.max().date()})")
    print(f"Val   : {len(df_val):>5}  "
          f"({df_val.Date.min().date()} → {df_val.Date.max().date()})")
    print(f"Test  : {len(df_test):>5}  "
          f"({df_test.Date.min().date()} → {df_test.Date.max().date()})")
    print(f"\nIMF8 range: train max={df_train['IMF8'].max():.0f}  "
          f"test max={df_test['IMF8'].max():.0f}  ← first-diff fix applied")
    print("=" * 65)

    # ── PRE-COMPUTE ONCE: scalers + GPU tensors for all 9 components ─────────
    # This is the key speed-up: windows are identical across all 50 seeds.
    # Doing this once saves ~49× redundant numpy + DataLoader work.
    scalers, imf8_meta, comp_windows = precompute_all_windows(
        df_train, df_val, df_test)
    print("=" * 65)

    # ── Checkpoint recovery ─────────────────────────────────────────────────
    completed_seeds, all_results = _load_checkpoint()
    _last_y_true = _last_y_hat = _last_comp_preds = None

    for seed in range(1, N_EXPERIMENTS + 1):

        # Skip already-completed seeds
        if seed in completed_seeds:
            print(f"[{seed:02d}/{N_EXPERIMENTS}] seed={seed}  ✓ skipped (checkpoint)",
                  flush=True)
            continue

        print(f"\n[{seed:02d}/{N_EXPERIMENTS}] seed={seed}", flush=True)
        m, cp, yt, yp = run_experiment(
            scalers, imf8_meta, comp_windows, df_test, seed=seed)
        m["experiment"] = seed
        m["seed"]       = seed
        all_results.append(m)

        # Persist immediately so a crash loses at most 1 seed
        _append_checkpoint(m)

        print(f"  MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}  "
              f"MAPE={m['MAPE']:.4f}%  R²={m['R2']:.4f}  "
              f"time={m['training_time_sec']:.1f}s", flush=True)
        _last_y_true     = yt
        _last_y_hat      = yp
        _last_comp_preds = cp

    # Sort results by seed for consistent summary output
    all_results.sort(key=lambda r: r["seed"])

    print("\n✅ All 50 experiments complete.")
    return df, df_test, all_results, _last_y_true, _last_y_hat


In [16]:
def cell14_save_csv(all_results):
    """
    Re-write the full CSV in seed order.
    (Individual rows were already appended live during Cell 13; this step
    ensures the final file is clean and sorted even after a resumed run.)
    """
    all_results_sorted = sorted(all_results, key=lambda r: r["seed"])
    with open(OUTPUT_CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=_CKPT_FIELDS, extrasaction="ignore")
        w.writeheader()
        w.writerows(all_results_sorted)
    print(f"Per-experiment CSV → {OUTPUT_CSV}")
    return pd.read_csv(OUTPUT_CSV)

In [17]:
def cell15_summary(all_results):
    """Print and save the 50-run aggregate statistics."""
    maes  = [r["MAE"]               for r in all_results]
    rmses = [r["RMSE"]              for r in all_results]
    mapes = [r["MAPE"]              for r in all_results]
    r2s   = [r["R2"]                for r in all_results]
    times = [r["training_time_sec"] for r in all_results]

    sep   = "=" * 72
    lines = [
        sep,
        "  CEEMDAN-Transformer  —  50-RUN STATISTICAL SUMMARY",
        sep,
        f'  {"Metric":<14} {"Mean":>12} {"± Std":>12} {"Min":>10} {"Max":>10}',
        "-" * 72,
    ]
    for label, arr in [("MAE", maes), ("RMSE", rmses),
                        ("MAPE (%)", mapes), ("R²", r2s), ("Time (s)", times)]:
        mn, sd, mi, mx = (np.mean(arr), np.std(arr),
                          np.min(arr), np.max(arr))
        lines.append(f"  {label:<14} {mn:>12.4f} {sd:>12.4f}"
                     f" {mi:>10.4f} {mx:>10.4f}")
    mn_t, sd_t = np.mean(times), np.std(times)
    lines += [
        "-" * 72,
        f'  {"Time (min)":<14} {mn_t/60:>12.4f} {sd_t/60:>12.4f}',
        sep,
    ]
    txt = "\n".join(lines)
    print("\n" + txt)
    with open(OUTPUT_TXT, "w") as f:
        f.write(txt + "\n")
    print(f"\nSummary → {OUTPUT_TXT}")
    return maes, rmses, mapes, r2s, times

In [18]:
def cell16_plot_actual_vs_predicted(df_test, y_true, y_hat):
    """Two-panel chart: price overlay + residual bar."""
    test_dates = df_test["Date"].values[:len(y_true)]

    fig, axes = plt.subplots(2, 1, figsize=(16, 10),
                              gridspec_kw={"height_ratios": [3, 1]})
    ax = axes[0]
    ax.plot(test_dates, y_true, color="#1565C0", lw=1.8, label="Actual Close")
    ax.plot(test_dates, y_hat,  color="#E53935", lw=1.3,
            ls="--", label="Predicted Close")
    ax.set_title(
        "CEEMDAN-Transformer: Actual vs Predicted (Test 2023–2025)",
        fontsize=14, fontweight="bold")
    ax.set_ylabel("NIFTY 50 Close")
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    errs = y_true - y_hat
    ax2.bar(test_dates, errs, color="#7B1FA2", alpha=0.6, width=1)
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_title("Prediction Error (Actual − Predicted)")
    ax2.set_xlabel("Date")
    ax2.set_ylabel("Error")
    ax2.grid(True, alpha=0.3)

    m50 = compute_metrics(y_true, y_hat)
    fig.text(0.01, 0.97,
             f'Seed 50  |  MAE={m50["MAE"]:.2f}  RMSE={m50["RMSE"]:.2f}  '
             f'MAPE={m50["MAPE"]:.4f}%  R²={m50["R2"]:.4f}',
             fontsize=10, va="top")
    plt.tight_layout()
    out1 = "/content/transformer_actual_vs_predicted.png"
    plt.savefig(out1, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plot saved → {out1}")

In [19]:
def cell17_metric_distributions(maes, rmses, mapes, r2s):
    """Four-panel histogram of MAE, RMSE, MAPE, R² across 50 seeds."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    data_pairs = [
        ("MAE",      maes,  "#1565C0"),
        ("RMSE",     rmses, "#C62828"),
        ("MAPE (%)", mapes, "#2E7D32"),
        ("R²",       r2s,   "#F57F17"),
    ]
    for ax, (label, arr, col) in zip(axes, data_pairs):
        ax.hist(arr, bins=15, color=col, alpha=0.8, edgecolor="white")
        ax.axvline(np.mean(arr), color="black", ls="--", lw=1.5,
                   label=f"Mean={np.mean(arr):.3f}")
        ax.set_title(label, fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    plt.suptitle(
        "Metric Distributions across 50 Independent Runs (CEEMDAN-Transformer)",
        fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    out2 = "/content/transformer_metric_distributions.png"
    plt.savefig(out2, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plot saved → {out2}")

In [20]:
def cell18_volatility_regime(df_test, y_true, y_hat):
    """Split test period into high / low volatility regimes and report metrics."""
    close_all = df_test[TARGET_COL].values.astype(np.float64)
    log_ret   = np.log(close_all[1:] / close_all[:-1])
    rol_vol   = pd.Series(log_ret).rolling(ROLLING_VOL_WINDOW).std()
    vol_vals  = rol_vol.values[:len(y_true)]
    vol_med   = float(np.nanmedian(vol_vals))
    vol_vals  = np.where(np.isnan(vol_vals), vol_med, vol_vals)

    high_mask = vol_vals >= vol_med
    low_mask  = ~high_mask
    m_h = compute_metrics(y_true[high_mask], y_hat[high_mask])
    m_l = compute_metrics(y_true[low_mask],  y_hat[low_mask])

    print("\n" + "=" * 66)
    print("  VOLATILITY REGIME VALIDATION (seed=50)")
    print(f"  30-day rolling vol | Median = {vol_med:.6f}")
    print("=" * 66)
    print(f'  {"Metric":<12} {"High Volatility":>18} {"Low Volatility":>18}')
    print("-" * 66)
    for k in ["MAE", "RMSE", "MAPE", "R2"]:
        print(f"  {k:<12} {m_h[k]:>18.4f} {m_l[k]:>18.4f}")
    print("=" * 66)
    print(f"  High-vol days : {high_mask.sum()}")
    print(f"  Low-vol  days : {low_mask.sum()}")

In [21]:
def main():
    df, df_test, all_results, y_true, y_hat = cell13_load_and_run()
    cell14_save_csv(all_results)
    maes, rmses, mapes, r2s, times         = cell15_summary(all_results)
    if y_true is not None and y_hat is not None:
        cell16_plot_actual_vs_predicted(df_test, y_true, y_hat)
        cell17_metric_distributions(maes, rmses, mapes, r2s)
        cell18_volatility_regime(df_test, y_true, y_hat)


if __name__ == "__main__":
    main()

Device  : cuda
PyTorch : 2.10.0+cu128
AMP     : enabled (FP16)
Running : 50 experiments  (seeds 1 → 50)
Train :  1770  (2015-01-09 → 2022-03-16)
Val   :   196  (2022-03-17 → 2022-12-30)
Test  :   633  (2023-01-02 → 2025-07-25)

IMF8 range: train max=16783  test max=23158  ← first-diff fix applied
  [Pre-compute] Fitting scalers & building GPU tensors …
    [IMF1] windowed
    [IMF2] windowed
    [IMF3] windowed
    [IMF4] windowed
    [IMF5] windowed
    [IMF6] windowed
    [IMF7] windowed
    [Residual] windowed
    [IMF8] windowed (first-diff)
  [Pre-compute] Done in 1.1s — tensors live on cuda
  ↻  Checkpoint found: 26 seeds already done (1–26).  Resuming…
[01/50] seed=1  ✓ skipped (checkpoint)
[02/50] seed=2  ✓ skipped (checkpoint)
[03/50] seed=3  ✓ skipped (checkpoint)
[04/50] seed=4  ✓ skipped (checkpoint)
[05/50] seed=5  ✓ skipped (checkpoint)
[06/50] seed=6  ✓ skipped (checkpoint)
[07/50] seed=7  ✓ skipped (checkpoint)
[08/50] seed=8  ✓ skipped (checkpoint)
[09/50] seed=9  ✓ sk

IndexError: boolean index did not match indexed array along axis 0; size of axis is 633 but size of corresponding boolean axis is 632